# Scatter and paired plots

In [ ]:
import os
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
import NLProcessing

date = datetime.today().strftime('%Y%m%d')

laptop = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"
homecomp = "C:\\Users\\user\\NUS Dropbox\\acclab\\Nicole M Lee"
labcomp = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee"
specifiedpath = labcomp

openPath = specifiedpath + "\\Data Compilation\\Climbing_New\\"
deltagdir = openPath + "Compilation with delta\\2025deltagcollection\\"

for f in font_manager.findSystemFonts(fontpaths=["fonts"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"
matplotlib.rcParams['svg.fonttype'] = 'none'

## Load

In [ ]:
files = os.listdir(deltagdir)
totalfile = pd.concat([pd.read_csv(deltagdir + n).assign(responder=n.split(" x ")[1].split("_")[0]) for n in files], ignore_index=True)
totalfile['genotypeandresponder'] = totalfile['MBON'] + "_" + totalfile['responder']

onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)
mbononly = onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4', 'R76B09', 'VT999036'])]

lobelocation = NLProcessing.generate_lobelocation(sorted({n.split(" ")[0] for n in files}), specifiedpath + "\\Data Compilation\\MBONlist.csv")

## Select responder

In [ ]:
cols_only = ['MBON', 'responder', 'genotypeandresponder', 'height_deltag', 'speed_deltag', 'bspeed_deltag',
             'maxvelocity_deltag', 'straightindex_deltag', 'meanbout_deltag', 'bout_deltag']

RESPONDERNAME = {'ACR': 'GtACR1', 'Chrimson2': 'CsChrimson'}

responder = "ACR"

df = (mbononly[mbononly['responder'] == responder][cols_only]
      .rename(columns=lambda c: c.replace('_deltag', ''))
      .merge(lobelocation[['MBON', 'Neurotransmitter']], on='MBON')
      .drop(columns=['MBON', 'responder', 'genotypeandresponder']))

## Scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
melted = pd.melt(df, id_vars=['Neurotransmitter'], var_name='Metrics', value_name='Δg')
g = sns.swarmplot(data=melted, x='Metrics', y='Δg', hue='Neurotransmitter', dodge=True)
sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
sns.set_style("darkgrid")
NLProcessing.wrap_labels(ax, 10)
g.set_title('Plot of MBONs > ' + RESPONDERNAME[responder] + ' and their Δg separated by neurotransmitter types according to locomotor metrics', weight='bold', fontsize=12)
plt.savefig(openPath + "images\\" + date + "_" + responder + "_separationbyNTtypes.svg", bbox_inches='tight')

## Paired plot

In [ ]:
t = sns.pairplot(data=df, hue='Neurotransmitter', hue_order=["Glutamate", "Acetylcholine", "GABA"])
t.fig.suptitle('MBONs > ' + RESPONDERNAME[responder], weight='bold', fontsize=16, y=0.99)
t.fig.legend(handles=t._legend_data.values(), labels=t._legend_data.keys(), loc='lower center', ncol=5)
plt.savefig(openPath + "images\\" + date + "_" + responder + "_pairplot.svg", bbox_inches='tight')